# Phân tích dữ liệu DUC_SUM

Notebook thống kê số file không có nội dung và số lượng thẻ mở `<s>` trong từng file.

## Tóm tắt

Chạy toàn bộ notebook để cập nhật kết quả theo dữ liệu hiện có trong `data/DUC_SUM`.

## Bối cảnh và phương pháp

### Giả định chính

- Một file được xem là **không có nội dung** nếu sau khi đọc và loại bỏ khoảng trắng, nội dung còn lại là chuỗi rỗng.
- Số tag `<s>` được tính bằng số thẻ mở có dạng `<s ...>` (không phân biệt chữ hoa/thường).
- Trung bình, lớn nhất và nhỏ nhất được tính trên **tất cả file**, bao gồm cả file rỗng. Notebook cũng hiển thị trung bình riêng trên các file có nội dung để tiện đối chiếu.

In [6]:
from pathlib import Path
import re
import statistics

def find_project_root(start_path):
    current_path = start_path.resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if (candidate_path / 'data' / 'DUC_SUM').is_dir():
            return candidate_path
    raise FileNotFoundError('Không tìm thấy thư mục data/DUC_SUM.')

project_root = find_project_root(Path.cwd())
duc_sum_dir = project_root / 'data' / 'DUC_SUM'
print(f'Thư mục dữ liệu: {duc_sum_dir}')

Thư mục dữ liệu: /Users/thangtran/Workplace/master_s_degree/nlp/nlp-practice/data/DUC_SUM


## Dữ liệu

Đọc toàn bộ file trực tiếp trong `data/DUC_SUM`, sắp xếp theo tên để kết quả luôn ổn định.

In [11]:
duc_sum_files = sorted(
    path for path in duc_sum_dir.iterdir()
    if path.is_file()
)

file_statistics = []

for file_path in duc_sum_files:
    file_content = file_path.read_text(
        encoding='utf-8',
        errors='replace',
    )

    has_content = bool(file_content.strip())

    sentence_contents = re.findall(
        r'<s\b[^>]*>(.*?)</s>',
        file_content,
        flags=re.IGNORECASE | re.DOTALL,
    )

    sentence_tag_count = len(sentence_contents)

    sentence_tag_character_counts = []

    for sentence_content in sentence_contents:
        character_count = len(sentence_content.strip())
        sentence_tag_character_counts.append(character_count)

    file_statistics.append({
        'file_name': file_path.name,
        'has_content': has_content,
        'sentence_tag_count': sentence_tag_count,
        'sentence_tag_character_counts': sentence_tag_character_counts,
        'sentence_contents': sentence_contents,
    })

print(f'Đã đọc {len(file_statistics)} file.')

Đã đọc 59 file.


## Kết quả

In [15]:
if not file_statistics:
    raise ValueError(f'Không có file nào trong {duc_sum_dir}.')

empty_files = [row for row in file_statistics if not row['has_content']]

all_tag_counts = [
    row['sentence_tag_count']
    for row in file_statistics
]

nonempty_tag_counts = [
    row['sentence_tag_count']
    for row in file_statistics
    if row['has_content']
]

files_with_sentence_tags = [
    row
    for row in file_statistics
    if row['sentence_tag_count'] > 0
]


# =========================
# Thống kê số lượng tag <s>
# =========================

total_tag_count = sum(all_tag_counts)

average_tag_count = total_tag_count / len(all_tag_counts)

median_tag_count = statistics.median(all_tag_counts)

quartiles = statistics.quantiles(
    all_tag_counts,
    n=4,
    method='inclusive',
)

first_quartile_tag_count = quartiles[0]
third_quartile_tag_count = quartiles[2]

tag_count_standard_deviation = statistics.pstdev(all_tag_counts)

maximum_tag_count = max(all_tag_counts)
minimum_tag_count = min(all_tag_counts)

average_nonempty_tag_count = (
    sum(nonempty_tag_counts) / len(nonempty_tag_counts)
    if nonempty_tag_counts
    else 0
)

minimum_nonempty_tag_count = (
    min(nonempty_tag_counts)
    if nonempty_tag_counts
    else 0
)

maximum_nonempty_tag_count = (
    max(nonempty_tag_counts)
    if nonempty_tag_counts
    else 0
)

maximum_files = [
    row['file_name']
    for row in file_statistics
    if row['sentence_tag_count'] == maximum_tag_count
]

minimum_files = [
    row['file_name']
    for row in file_statistics
    if row['sentence_tag_count'] == minimum_tag_count
]


# ===============================
# Thống kê độ dài nội dung tag <s>
# ===============================

all_tag_character_counts = []
shortest_sentence = None
longest_sentence = None


for row in file_statistics:
    for index in range(len(row['sentence_contents'])):
        sentence_content = row['sentence_contents'][index]
        character_count = row['sentence_tag_character_counts'][index]

        if (
            shortest_sentence is None
            or character_count < shortest_sentence['character_count']
        ):
            shortest_sentence = {
                'file_name': row['file_name'],
                'character_count': character_count,
                'content': sentence_content,
            }

        if (
            longest_sentence is None
            or character_count > longest_sentence['character_count']
        ):
            longest_sentence = {
                'file_name': row['file_name'],
                'character_count': character_count,
                'content': sentence_content,
            }

if all_tag_character_counts:
    total_tag_characters = sum(all_tag_character_counts)

    average_tag_character_count = (
        total_tag_characters / len(all_tag_character_counts)
    )

    minimum_tag_character_count = min(all_tag_character_counts)
    maximum_tag_character_count = max(all_tag_character_counts)
else:
    total_tag_characters = 0
    average_tag_character_count = 0
    minimum_tag_character_count = 0
    maximum_tag_character_count = 0


# =========================
# Print kết quả
# =========================

print(f'Tổng số file: {len(file_statistics)}')
print(f'Số file không có nội dung: {len(empty_files)}')
print(f'Số file có nội dung: {len(file_statistics) - len(empty_files)}')
print(f'Số file có ít nhất một tag <s>: {len(files_with_sentence_tags)}')

print()

print(f'Tổng số tag <s>: {total_tag_count}')
print(f'Số tag <s> trung bình (tất cả file): {average_tag_count:.2f}')
print(
    f'Số tag <s> trung bình (file có nội dung): '
    f'{average_nonempty_tag_count:.2f}'
)
print(f'Trung vị số tag <s>: {median_tag_count:.2f}')
print(
    f'Độ lệch chuẩn số tag <s>: '
    f'{tag_count_standard_deviation:.2f}'
)

maximum_file_names = ', '.join(maximum_files)
minimum_file_names = ', '.join(minimum_files)

print(
    f'Số tag <s> nhiều nhất: '
    f'{maximum_tag_count} - file: {maximum_file_names}'
)

print(
    f'Số tag <s> thấp nhất: '
    f'{minimum_tag_count} - file: {minimum_file_names}'
)

print(
    f'Số tag <s> thấp nhất trong file có nội dung: '
    f'{minimum_nonempty_tag_count}'
)

print(
    f'Số tag <s> nhiều nhất trong file có nội dung: '
    f'{maximum_nonempty_tag_count}'
)

if shortest_sentence is not None:
    print()
    print('--- Tag <s> ngắn nhất ---')
    print(f"File: {shortest_sentence['file_name']}")
    print(f"Số ký tự: {shortest_sentence['character_count']}")
    print(f"Nội dung: {shortest_sentence['content']}")

if longest_sentence is not None:
    print()
    print('--- Tag <s> dài nhất ---')
    print(f"File: {longest_sentence['file_name']}")
    print(f"Số ký tự: {longest_sentence['character_count']}")
    print(f"Nội dung: {longest_sentence['content']}")

Tổng số file: 59
Số file không có nội dung: 16
Số file có nội dung: 43
Số file có ít nhất một tag <s>: 42

Tổng số tag <s>: 620
Số tag <s> trung bình (tất cả file): 10.51
Số tag <s> trung bình (file có nội dung): 14.42
Trung vị số tag <s>: 13.00
Độ lệch chuẩn số tag <s>: 7.30
Số tag <s> nhiều nhất: 21 - file: d115i
Số tag <s> thấp nhất: 0 - file: d061j, d062j, d067f, d068f, d071f, d072f, d074b, d078b, d084a, d085d, d086d, d090d, d097e, d098e, d104g, d106g, d108g
Số tag <s> thấp nhất trong file có nội dung: 0
Số tag <s> nhiều nhất trong file có nội dung: 21

--- Tag <s> ngắn nhất ---
File: d093c
Số ký tự: 4
Nội dung:  U.S.

--- Tag <s> dài nhất ---
File: d117i
Số ký tự: 524
Nội dung:  The shortlist for the Booker, the UK's most hyped literary prize and one of the most lucrative, is all the more surprising in a bumper year for new fiction fulfilling the criteria - English language and non-American - for consideration for the award.
<s docid="FT944-16774" num="6" wdcount="42"> After hours

### Danh sách file không có nội dung

In [ ]:
if empty_files:
    for row in empty_files:
        print(f"- {row['file_name']}")
else:
    print('Không có file rỗng.')

### Chi tiết số tag `<s>` theo file

In [4]:
print(f"{'Tên file':<12} {'Có nội dung':<14} {'Số tag <s>':>10}")
print('-' * 38)
for row in file_statistics:
    content_status = 'Có' if row['has_content'] else 'Không'
    print(f"{row['file_name']:<12} {content_status:<14} {row['sentence_tag_count']:>10}")

Tên file     Có nội dung    Số tag <s>
--------------------------------------
d061j        Có                     18
d062j        Không                   0
d063j        Có                     18
d064j        Có                     19
d065j        Có                     12
d066j        Có                     19
d067f        Không                   0
d068f        Không                   0
d069f        Có                     14
d070f        Có                     16
d071f        Không                   0
d072f        Không                   0
d073b        Có                     12
d074b        Không                   0
d075b        Có                     14
d076b        Có                     13
d077b        Có                     14
d078b        Không                   0
d079a        Có                     14
d080a        Có                     11
d081a        Có                     17
d082a        Có                     13
d083a        Có                     17
d084a        Không       

## Kết luận

Kết quả phía trên được tính trực tiếp từ dữ liệu mỗi lần chạy notebook. Giá trị nhỏ nhất trên tất cả file có thể bằng `0` vì các file không có nội dung vẫn thuộc tập dữ liệu phân tích.